# Prueba final en Colab T4 limpio (Persona B)

Simula el entorno del profe **antes** de entregar (solo hay 1 corrección en 24 h, con −10).

Checklist que valida este notebook:
1. Repo clonado + `pip install -r requirements.txt` en un Colab limpio.
2. MySQL 8.4.10 (binario Linux) con usuario de solo lectura `lector` y `ONLY_FULL_GROUP_BY`.
3. `import solucion` < **5 min**; cada `generar_sql` < **30 s** (también con autocorrección).
4. Dos corridas de la validación → salidas **idénticas** (determinismo).
5. `usar_adapter: false` funciona con el mismo prompt.
6. `pip freeze` de lo necesario → `requirements.txt` definitivo.

**Entorno → Cambiar tipo de entorno de ejecución → GPU T4.** Correr de arriba a abajo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

## 1. Repo y dependencias

In [ ]:
REPO = "https://github.com/jcdiegolopez/DP-PR3.git"
RAMA = "main"
!rm -rf /content/DP-PR3 && git clone -q --branch $RAMA $REPO /content/DP-PR3
%cd /content/DP-PR3
!git log --oneline -3
!ls -la adapter/

In [ ]:
import time
t = time.time()
!pip install -q -r requirements.txt pymysql
# Colab trae torchao 0.10, que peft 0.21 rechaza con ImportError; no se usa aquí.
!pip uninstall -q -y torchao
print(f"pip install: {time.time() - t:.0f}s")

## 2. MySQL 8.4.10 (binario Linux)

Reemplazar `MYSQL_URL` por el enlace exacto del PDF del proyecto si es distinto.

In [ ]:
MYSQL_URL = "https://dev.mysql.com/get/Downloads/MySQL-8.4/mysql-8.4.10-linux-glibc2.28-x86_64.tar.xz"
!apt-get -qq install -y libaio1t64 libnuma1 > /dev/null 2>&1 || apt-get -qq install -y libaio1 libnuma1 > /dev/null 2>&1
!ln -sf /usr/lib/x86_64-linux-gnu/libaio.so.1t64 /usr/lib/x86_64-linux-gnu/libaio.so.1 2>/dev/null; true
!wget -q -O /tmp/mysql.tar.xz "$MYSQL_URL" && mkdir -p /opt/mysql && tar -xf /tmp/mysql.tar.xz -C /opt/mysql --strip-components=1
!/opt/mysql/bin/mysqld --version

In [ ]:
import subprocess, os, time
os.makedirs("/opt/mysql-data", exist_ok=True)
!/opt/mysql/bin/mysqld --initialize-insecure --user=root --datadir=/opt/mysql-data/data --basedir=/opt/mysql 2>&1 | tail -2
mysqld = subprocess.Popen(["/opt/mysql/bin/mysqld", "--user=root", "--datadir=/opt/mysql-data/data",
                           "--basedir=/opt/mysql", "--socket=/tmp/mysql.sock", "--port=3306",
                           "--log-error=/tmp/mysqld.err"])
for _ in range(60):
    if os.path.exists("/tmp/mysql.sock"):
        break
    time.sleep(1)
MYSQL = "/opt/mysql/bin/mysql -uroot --socket=/tmp/mysql.sock"
!$MYSQL -e "CREATE USER IF NOT EXISTS 'lector'@'%' IDENTIFIED BY 'lector'; GRANT SELECT, SHOW VIEW ON *.* TO 'lector'@'%'; SELECT VERSION(), @@sql_mode;"

In [ ]:
# Esquemas de validación de C (+ datos). Cada .sql debe crear su propia base (CREATE DATABASE / USE).
import glob
for f in sorted(glob.glob("datos/esquemas/validacion/*.sql")):
    print(f)
    !$MYSQL < "$f"
!$MYSQL -e "SHOW DATABASES"

## 3. Carga del modelo (< 5 min)

In [ ]:
import json, time, pymysql
t0 = time.time()
import solucion
CARGA = time.time() - t0
print(f"carga: {CARGA:.1f}s  (límite 300s)  config={solucion.CFG}  device={solucion.DEVICE}")
assert CARGA < 300

In [ ]:
preguntas = [json.loads(l) for l in open("datos/validacion_propia.jsonl", encoding="utf-8")]
print(len(preguntas), "preguntas")


def esquema_de(db):
    # Same text the grader gives: SHOW CREATE TABLE of every table.
    con = pymysql.connect(unix_socket="/tmp/mysql.sock", user="lector", password="lector", database=db)
    with con.cursor() as cur:
        cur.execute("SHOW TABLES"); tablas = [r[0] for r in cur.fetchall()]
        partes = []
        for t in tablas:
            cur.execute(f"SHOW CREATE TABLE `{t}`"); partes.append(cur.fetchone()[1])
    con.close()
    return "\n\n".join(partes)


def hacer_ejecutar(db):
    def ejecutar(sql):
        try:
            con = pymysql.connect(unix_socket="/tmp/mysql.sock", user="lector", password="lector",
                                  database=db, read_timeout=10)
            with con.cursor() as cur:
                cur.execute(sql)
                cols = [d[0] for d in cur.description] if cur.description else []
                filas = [list(r) for r in cur.fetchmany(10)]
            con.close()
            return {"ok": True, "columnas": cols, "filas": filas, "error": None}
        except Exception as e:
            return {"ok": False, "columnas": [], "filas": [], "error": str(e)}
    return ejecutar

## 4. Tiempos y determinismo (dos corridas, sin y con autocorrección)

In [ ]:
def corrida(con_ejecutar):
    salidas, tiempos = [], []
    for p in preguntas:
        esquema = p.get("esquema") or esquema_de(p["db_id"])
        t = time.time()
        sql = solucion.generar_sql(esquema, p["pregunta"],
                                   ejecutar=hacer_ejecutar(p["db_id"]) if con_ejecutar else None)
        tiempos.append(time.time() - t); salidas.append(sql)
    return salidas, tiempos

for con_ejecutar in (False, True):
    s1, t1 = corrida(con_ejecutar)
    s2, t2 = corrida(con_ejecutar)
    print(f"ejecutar={con_ejecutar}: max {max(t1 + t2):.1f}s, media {sum(t1) / len(t1):.1f}s, "
          f"idénticas={s1 == s2}")
    assert max(t1 + t2) < 30, "una pregunta pasó de 30 s"
    assert s1 == s2, [i for i, (a, b) in enumerate(zip(s1, s2)) if a != b]

In [ ]:
validas = sum(hacer_ejecutar(p["db_id"])(s)["ok"] for p, s in zip(preguntas, s1))
print(f"{validas}/{len(preguntas)} consultas corren en MySQL 8.4 (con autocorrección)")
for p, s in list(zip(preguntas, s1))[:5]:
    print(p["pregunta"], "\n  ", s)

## 5. `usar_adapter: false` (editando `config.json` como lo haría el profe)

In [ ]:
import shutil, subprocess
shutil.copy("config.json", "/tmp/config.json.bak")
cfg = json.load(open("config.json")); cfg["usar_adapter"] = False
json.dump(cfg, open("config.json", "w"), indent=2)
codigo = '''
import json, time
t = time.time(); import solucion; print("carga", round(time.time() - t, 1), "s", solucion.CFG["usar_adapter"])
p = json.loads(open("datos/validacion_propia.jsonl", encoding="utf-8").readline())
t = time.time(); print(solucion.generar_sql(p["esquema"], p["pregunta"]), round(time.time() - t, 1), "s")
'''
print(subprocess.run([sys.executable, "-c", codigo], capture_output=True, text=True).stdout)
shutil.copy("/tmp/config.json.bak", "config.json")
!git diff --stat config.json

## 6. Versiones que funcionaron → `requirements.txt`

In [ ]:
!pip freeze | grep -iE "^(torch|transformers|peft|accelerate|safetensors|tokenizers|huggingface.hub)=="
print("\nCopiar estas versiones a requirements.txt (torch puede quedar como >= para no forzar reinstalación).")